# L13 — Data preparation pipelines: dedup, PII, data cards (Spark + Delta)
**Objective 16.**

**Northfield Grocers context:** Supplier sheets and catalog extracts feed both the extraction fine-tune and its eval set. Northfield's pipeline must remove duplicate listings, quarantine bad taxonomy labels, redact supplier rep contact details, and guarantee no eval item leaked into training — then write Delta tables with a data card the governance team can read.

**Retail use cases:** Preparing supplier-sheet corpora for extraction fine-tuning; eval-set hygiene before a model swap; supplier confidentiality.

**Platform:** Databricks (no GPU). Runs in local Spark here; Delta write is exercised on Databricks.

**Done means:** all planted defects (D1–D5) are detected and removed; leakage check passes; data card generated.

## Step 1 — Load with Spark

In [1]:
# === Lab environment header (identical in every lab) ===
import os, sys, json, time, math, shutil, re, subprocess, importlib, importlib.util
import numpy as np, pandas as pd

def ensure_packages(pkgs):
    """Install any missing pip packages into THIS Python (same mechanism as %pip on Databricks) and import them.
    Fresh packages are importable immediately — no restart. Only labs that need extras call this (L02, L08)."""
    missing = [p for p in pkgs if importlib.util.find_spec(p.replace("-", "_")) is None]
    if not missing:
        print("Packages present:", pkgs); return
    print("Installing missing packages into", sys.executable, ":", missing)
    cmd = [sys.executable, "-m", "pip", "install", "-q", *missing]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        r = subprocess.run(cmd + ["--break-system-packages"], capture_output=True, text=True)   # local system Pythons
    if r.returncode != 0:
        raise ImportError("pip could not install " + str(missing) + ". Ask the admin to add them as cluster libraries "
                          "(Compute → Libraries → PyPI) or use an internal index. pip said: " + r.stderr[-600:])
    importlib.invalidate_caches()
    for p in missing: importlib.import_module(p.replace("-", "_"))
    print("Installed and imported:", missing)

# Mode: "GPU" runs the full lab on Azure GPU compute; "SMOKE" runs the CPU/synthetic path anywhere.
LAB_MODE = os.environ.get("LAB_MODE") or ("GPU" if shutil.which("nvidia-smi") else "SMOKE")

# Data folder: env override → package-relative (../../data) → Unity Catalog volume → search the workspace once
_candidates = [os.environ.get("DATA_DIR"), os.path.abspath(os.path.join(os.getcwd(), "..", "..", "data")), "/Volumes/northfield/llmops/labdata"]
DATA_DIR = next((c for c in _candidates if c and os.path.exists(os.path.join(c, "catalog_items.csv"))), None)
if DATA_DIR is None:
    import glob
    _hits = [h for root in ("/Workspace", "/Volumes", os.path.expanduser("~")) if os.path.isdir(root)
             for h in glob.glob(os.path.join(root, "**", "catalog_items.csv"), recursive=True)][:1]
    DATA_DIR = os.path.dirname(_hits[0]) if _hits else None
if DATA_DIR is None:
    raise FileNotFoundError("Lab data not found. Upload the package's data/ folder to a Unity Catalog volume and set "
                            "os.environ['DATA_DIR'] = '/Volumes/<catalog>/<schema>/<volume>' in a cell above this one.")
def gpu_only(msg):
    """Called wherever a step needs a GPU / model download that the smoke path cannot run."""
    print(f"[{LAB_MODE}] GPU-only step not executed here: {msg}")
def check(cond, msg):
    """Binary 'done means' assertion — prints PASS/FAIL and raises on FAIL so the notebook stops."""
    print(("PASS " if cond else "FAIL ") + msg); assert cond, msg
print(f"LAB_MODE={LAB_MODE}  DATA_DIR={DATA_DIR}  python={sys.version.split()[0]}")

LAB_MODE=SMOKE  DATA_DIR=/home/claude/llmops_labs/data  python=3.12.3


In [2]:
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.master(os.environ.get("SPARK_MASTER", "local[1]")).appName("L13").config("spark.ui.enabled", "false").getOrCreate()
cat = spark.read.csv(os.path.join(DATA_DIR, "catalog_items.csv"), header=True, inferSchema=True)
sup = spark.read.csv(os.path.join(DATA_DIR, "supplier_descriptions.csv"), header=True)
n_cat0, n_sup0 = cat.count(), sup.count(); print(n_cat0, "catalog rows;", n_sup0, "supplier rows")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/12 08:24:12 WARN Utils: Your hostname, vm, resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/09/12 08:24:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/usr/local/lib/python3.12/dist-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


26/09/12 08:24:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


3065 catalog rows; 1200 supplier rows


## Step 2 — Exact and near-duplicate removal (D1, D2)
*Why:* duplicates in a fine-tune corpus over-weight examples; duplicates across train/eval inflate scores. Exact first (item_id + title), then a normalised key (lower, collapse whitespace).

In [3]:
cat1 = cat.dropDuplicates(["item_id", "title"])
cat1 = cat1.withColumn("title_norm", F.lower(F.trim(F.regexp_replace("title", r"\s+", " "))))
cat2 = cat1.dropDuplicates(["title_norm"])
n1, n2 = cat1.count(), cat2.count()
print(f"exact dupes removed: {n_cat0-n1}; near-dupes removed: {n1-n2}")
check(n_cat0 - n1 == 40 and n1 - n2 >= 25, "D1 (40 exact) and D2 (≥25 near) removed")

exact dupes removed: 40; near-dupes removed: 438
PASS D1 (40 exact) and D2 (≥25 near) removed


## Step 3 — Label validation (D4)
*Why:* an out-of-vocabulary label silently becomes a new class. Enforce the allowed set and quarantine the rest.

In [4]:
ALLOWED = ["Dairy", "Bakery", "Produce", "Frozen", "Beverages", "Snacks", "Household", "Personal Care"]
quarantine = cat2.filter(~F.col("category").isin(ALLOWED)); cat3 = cat2.filter(F.col("category").isin(ALLOWED))
nq = quarantine.count(); print("quarantined:", nq)
check(nq >= 1 and cat3.filter(F.col("category") == "UNKNOWN_CAT").count() == 0, "D4 quarantined")

quarantined: 13


PASS D4 quarantined


## Step 4 — PII redaction (D3)
*Why:* supplier notes leak contact details; a fine-tuned model will happily reproduce them. Regex redaction for emails and phone numbers is the minimum; log the count so the data card can state it.

In [5]:
EMAIL = r"[\w.+-]+@[\w-]+\.[\w.-]+"; PHONE = r"\+?\d[\d\-\s]{7,}\d"
sup1 = sup.withColumn("had_pii", F.col("description").rlike(EMAIL) | F.col("description").rlike(PHONE))
sup2 = sup1.withColumn("description", F.regexp_replace(F.regexp_replace("description", EMAIL, "[EMAIL]"), PHONE, "[PHONE]"))
n_pii = sup2.filter("had_pii").count(); leftover = sup2.filter(F.col("description").rlike(EMAIL) | F.col("description").rlike(PHONE)).count()
print("PII rows redacted:", n_pii, "| leftover matches:", leftover)
check(n_pii == 30 and leftover == 0, "D3: all 30 PII rows redacted")

PII rows redacted: 30 | leftover matches: 0
PASS D3: all 30 PII rows redacted


## Step 5 — Train/eval leakage check (D5)
*Why:* ten eval prompts were planted verbatim in train. Detect them with a join on the prompt text and remove them from eval (never from train — eval must stay clean and independent).

In [6]:
tr = spark.read.json(os.path.join(DATA_DIR, "train_prompts.jsonl")); ev = spark.read.json(os.path.join(DATA_DIR, "eval_prompts.jsonl"))
leaked = ev.join(tr.select("prompt"), "prompt", "inner"); ev_clean = ev.join(tr.select("prompt"), "prompt", "left_anti")
nl, ne = leaked.count(), ev_clean.count(); print("leaked eval items:", nl, "| clean eval size:", ne)
check(nl == 10 and ne == ev.count() - 10, "D5: 10 leaked items removed from eval")

leaked eval items: 10 | clean eval size: 190
PASS D5: 10 leaked items removed from eval


## Step 6 — Write Delta and the data card
*Why:* Delta gives versioned, ACID tables in Unity Catalog; the data card records provenance and every transformation count. On Databricks write `format("delta")` to `northfield.llmops.catalog_clean`; locally we write Parquet and keep the same card.

In [7]:
out = "/tmp/l13"; os.makedirs(out, exist_ok=True)
fmt = "delta" if os.environ.get("DATABRICKS_RUNTIME_VERSION") else "parquet"
cat3.drop("title_norm").write.mode("overwrite").format(fmt).save(f"{out}/catalog_clean")
sup2.drop("had_pii").write.mode("overwrite").format(fmt).save(f"{out}/supplier_clean")
card = dict(dataset="Northfield Grocers catalog + supplier sheets (synthetic)", built=time.strftime("%Y-%m-%d"), source_rows=dict(catalog=n_cat0, supplier=n_sup0),
            transformations=dict(exact_dupes_removed=n_cat0 - n1, near_dupes_removed=n1 - n2, labels_quarantined=nq, pii_rows_redacted=n_pii, eval_leakage_removed=nl),
            final_rows=dict(catalog=cat3.count(), supplier=sup2.count(), eval=ne), intended_use="fine-tune & eval for attribute extraction", not_for="any production decision — synthetic data")
json.dump(card, open(f"{out}/data_card.json", "w"), indent=2); print(json.dumps(card, indent=2))
check(all(v > 0 for v in card["transformations"].values()), "data card records every transformation")
spark.stop(); print("L13 complete.")

{
  "dataset": "Northfield Grocers catalog + supplier sheets (synthetic)",
  "built": "2026-09-12",
  "source_rows": {
    "catalog": 3065,
    "supplier": 1200
  },
  "transformations": {
    "exact_dupes_removed": 40,
    "near_dupes_removed": 438,
    "labels_quarantined": 13,
    "pii_rows_redacted": 30,
    "eval_leakage_removed": 10
  },
  "final_rows": {
    "catalog": 2574,
    "supplier": 1200,
    "eval": 190
  },
  "intended_use": "fine-tune & eval for attribute extraction",
  "not_for": "any production decision \u2014 synthetic data"
}
PASS data card records every transformation


L13 complete.
